# 01 — Perfil inicial del dataset

**Objetivo:** Realizar la comprensión inicial de las tres tablas modelables del proyecto (`ventas_crudas`, `dim_producto`, `ventas_semanales`), describiendo sus dimensiones, tipos de datos, calidad básica y características generales.

**Fase CRISP-DM:** 2 — Comprensión de los datos.

**Inputs:**
- Tabla `ventas_crudas` (MySQL)
- Tabla `dim_producto` (MySQL)
- Tabla `ventas_semanales` (MySQL)

**Outputs:**
- Insights iniciales documentados en este notebook.
- Confirmación de que las tablas están listas para el EDA profundo (`02_eda.ipynb`).

**Lo que NO se hace en este notebook:**
- Visualizaciones (van en `02_eda.ipynb`).
- Clasificación ABC/XYZ (va en `04_abc_xyz.ipynb`).
- Modelado predictivo (va en `05+`).

**Autor:** Equipo del proyecto — Diego Andrés De Jesús Montenegro y Luis David Andrade Díaz
**Fecha:** mayo 2026

In [1]:
# =========================
# Imports
# =========================
import sys
from pathlib import Path

# Permitir importar modulos de src/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from sqlalchemy import text

from src.db import get_engine

In [2]:
# =========================
# Configuracion
# =========================
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

engine = get_engine()
print("Conexion a MySQL establecida.")

Conexion a MySQL establecida.


## 1. Carga de las tres tablas modelables

Se cargan completas las tres tablas desde MySQL. Por el tamaño moderado (<600K filas), no es necesario hacer lectura por chunks.

In [3]:
# Cargar las tres tablas
ventas_crudas = pd.read_sql("SELECT * FROM ventas_crudas", engine, parse_dates=["fecha"])
dim_producto = pd.read_sql("SELECT * FROM dim_producto", engine,
                            parse_dates=["fecha_primera_venta", "fecha_ultima_venta"])
ventas_semanales = pd.read_sql("SELECT * FROM ventas_semanales", engine,
                                parse_dates=["fecha_inicio_semana"])

print(f"ventas_crudas:    {ventas_crudas.shape[0]:>8,} filas x {ventas_crudas.shape[1]:>2} columnas")
print(f"dim_producto:     {dim_producto.shape[0]:>8,} filas x {dim_producto.shape[1]:>2} columnas")
print(f"ventas_semanales: {ventas_semanales.shape[0]:>8,} filas x {ventas_semanales.shape[1]:>2} columnas")

ventas_crudas:     579,485 filas x 16 columnas
dim_producto:        4,282 filas x 12 columnas
ventas_semanales:  184,200 filas x 10 columnas


## 2. Tabla `ventas_crudas`

Datos transaccionales tipificados y limpios. Cada fila representa una línea de venta efectiva. Es la base para construir las demás tablas y el insumo histórico del proyecto.

In [ ]:
# Vistazo inicial
print("Primeras 5 filas:")
display(ventas_crudas.head())

print("\nTipos de datos:")
print(ventas_crudas.dtypes)

print(f"\nMemoria utilizada: {ventas_crudas.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")

In [4]:
# Estadisticas descriptivas de las variables numericas
ventas_crudas[["cantidad", "precio_unitario", "valor_bruto", "valor_costo"]].describe()

,cantidad,precio_unitario,valor_bruto,valor_costo
count,"579,485.0000","579,485.0000","579,485.0000","579,485.0000"
mean,40.8776,"27,321.3253","263,321.8567","237,471.4935"
std,320.5710,"250,882.3687","1,949,003.1480","1,939,026.5009"
min,0.0100,0.7200,0.0000,"-10,971.3800"
25%,1.0000,"3,000.0000","6,639.0000","4,731.4200"
50%,3.0000,"7,600.0000","23,530.0000","17,974.5300"
75%,10.0000,"30,000.0000","94,538.0000","76,889.0900"
max,"50,000.0000","163,607,000.0000","572,067,227.0000","526,841,213.2000"


In [5]:
# Calidad: nulos por columna
nulos = ventas_crudas.isna().sum()
nulos_pct = (nulos / len(ventas_crudas) * 100).round(2)
calidad = pd.DataFrame({"nulos": nulos, "porcentaje": nulos_pct})
calidad[calidad["nulos"] > 0].sort_values("nulos", ascending=False)

,nulos,porcentaje
proveedor_codigo,71087,12.2700
proveedor_nombre,71087,12.2700
nombre_linea_n1,314,0.0500
nombre_linea_n2,314,0.0500


In [6]:
# Cobertura temporal
print(f"Fecha minima: {ventas_crudas['fecha'].min().date()}")
print(f"Fecha maxima: {ventas_crudas['fecha'].max().date()}")
print(f"Dias unicos:  {ventas_crudas['fecha'].nunique():,}")

# Conteo por anio
por_anio = ventas_crudas.groupby(ventas_crudas["fecha"].dt.year).size()
por_anio.name = "filas"
por_anio.to_frame()

Fecha minima: 2024-01-03
Fecha maxima: 2026-03-31
Dias unicos:  659


,filas
fecha,
2024,236773
2025,272017
2026,70695


In [7]:
# Cardinalidad de variables categoricas
categoricas = ["item", "centro_operacion", "nombre_linea_n1", "nombre_linea_n2",
               "proveedor_codigo", "unidad_inventario"]

cardinalidad = pd.DataFrame({
    "columna": categoricas,
    "valores_unicos": [ventas_crudas[c].nunique() for c in categoricas],
})
cardinalidad

,columna,valores_unicos
0,item,4282
1,centro_operacion,3
2,nombre_linea_n1,14
3,nombre_linea_n2,72
4,proveedor_codigo,89
5,unidad_inventario,4


## 3. Tabla `dim_producto`

Una fila por SKU (4,282 productos). Es la dimensión del portafolio, con metadatos de cada producto: línea, proveedor, fechas de actividad.

In [8]:
# Vistazo a dim_producto
print("Primeras 5 filas:")
display(dim_producto.head())

print(f"\nTotal SKUs: {len(dim_producto):,}")
print(f"SKUs activos (con venta en los ultimos 90 dias): {dim_producto['activo'].sum():,}")
print(f"SKUs inactivos: {(dim_producto['activo'] == 0).sum():,}")

Primeras 5 filas:


,item,nombre_item,referencia_item,unidad_inventario,proveedor_codigo,proveedor_nombre,nombre_linea_n1,nombre_linea_n2,fecha_primera_venta,fecha_ultima_venta,activo,created_at
0,000001,SIKA EMULSION ASFALTICA GLN * 3.5 KLS,94593,UND,860000896,SIKA COLOMBIA SAS,SIKA,PORTAFOLIO,2024-01-03,2026-03-31,1,2026-05-18 04:16:11
1,000002,SIKA ESTUCO ACRILICO 1/4 * 1.5 KLS,94355,UND,860000896,SIKA COLOMBIA SAS,SIKA,PORTAFOLIO,2024-01-03,2026-03-31,1,2026-05-18 04:16:11
2,000003,SIKA ESTUCO ACRILICO GLN * 6.2 KLS,94357,UND,860000896,SIKA COLOMBIA SAS,SIKA,PORTAFOLIO,2024-01-03,2026-03-31,1,2026-05-18 04:16:11
3,000004,SIKA ESTUKA DOS * KILO,01E005,KIL,860000896,SIKA COLOMBIA SAS,SIKA,PORTAFOLIO,2024-01-03,2026-03-31,1,2026-05-18 04:16:11
4,000005,SIKA ESTUCO ACRILICO CUNETE * 30 KLS,545226,UND,860000896,SIKA COLOMBIA SAS,SIKA,MASIVOS,2024-01-03,2026-03-31,1,2026-05-18 04:16:11



Total SKUs: 4,282
SKUs activos (con venta en los ultimos 90 dias): 2,310
SKUs inactivos: 1,972


In [9]:
# Distribucion por linea principal (N1)
print("Top 10 lineas principales (N1) por numero de SKUs:")
dim_producto["nombre_linea_n1"].value_counts().head(10).to_frame("num_skus")

Top 10 lineas principales (N1) por numero de SKUs:


,num_skus
nombre_linea_n1,
OBRA BLANCA Y LADRILLOS,1024
HERRAMIENTAS,680
PVC-HD,565
GRIFERIA Y PLOMERIA,472
HOGAR,275
SIKA,226
PINTURAS,204
ELECTRICOS,189
BAJA ROTACION,182


In [10]:
# Top proveedores por numero de SKUs
print("Top 10 proveedores por numero de SKUs:")
dim_producto["proveedor_nombre"].value_counts().head(10).to_frame("num_skus")

Top 10 proveedores por numero de SKUs:


,num_skus
proveedor_nombre,
DISTRIBUCIONES HOYOSTOOLS SA,528
CIA COLOMBIANA DE CERAMICA SA,452
MEXICHEM COLOMBIA SAS,439
SIKA COLOMBIA SAS,232
PINTUCO COLOMBIA SAS,173
SEGAR SA,154
ALFAGRES SA,119
SANTIAGO PLAZA SA,68
ETEX COLOMBIA SA,55


In [11]:
# Cuanto tiempo lleva cada SKU activo
dim_producto["dias_de_historia"] = (
    dim_producto["fecha_ultima_venta"] - dim_producto["fecha_primera_venta"]
).dt.days

print("Distribucion de dias de historia por SKU:")
dim_producto["dias_de_historia"].describe().to_frame()

Distribucion de dias de historia por SKU:


,dias_de_historia
count,"4,282.0000"
mean,438.5096
std,321.4745
min,0.0000
25%,94.2500
50%,480.0000
75%,784.7500
max,818.0000


## 4. Tabla `ventas_semanales`

Granularidad de modelado: una fila por (`item` × `centro_operacion` × semana ISO). Es el dataset principal que alimenta los modelos de forecasting.

In [12]:
# Vistazo a ventas_semanales
print("Primeras 5 filas:")
display(ventas_semanales.head())

print(f"\nTotal filas: {len(ventas_semanales):,}")
print(f"SKUs distintos: {ventas_semanales['item'].nunique():,}")
print(f"Centros: {ventas_semanales['centro_operacion'].nunique()}")
print(f"Semanas distintas: {ventas_semanales['fecha_inicio_semana'].nunique()}")

Primeras 5 filas:


,id,item,centro_operacion,anio,semana,fecha_inicio_semana,cantidad_total,valor_bruto_total,num_transacciones,created_at
0,1,000001,001,2024,1,2024-01-01,1.0000,"29,663.0000",1,2026-05-18 04:16:12
1,2,000001,001,2024,2,2024-01-08,1.0000,"28,319.0000",1,2026-05-18 04:16:12
2,3,000001,001,2024,3,2024-01-15,5.0000,"128,571.0000",1,2026-05-18 04:16:12
3,4,000001,001,2024,6,2024-02-05,2.0000,"59,328.0000",2,2026-05-18 04:16:12
4,5,000001,001,2024,7,2024-02-12,3.0000,"84,286.0000",3,2026-05-18 04:16:12



Total filas: 184,200
SKUs distintos: 4,282
Centros: 3
Semanas distintas: 118


In [13]:
# Cobertura del grid teorico
n_skus = ventas_semanales["item"].nunique()
n_centros = ventas_semanales["centro_operacion"].nunique()
n_semanas = ventas_semanales["fecha_inicio_semana"].nunique()
combinaciones_teoricas = n_skus * n_centros * n_semanas
combinaciones_reales = len(ventas_semanales)

print(f"Combinaciones teoricas (SKU x centro x semana): {combinaciones_teoricas:>10,}")
print(f"Combinaciones con venta:                        {combinaciones_reales:>10,}")
print(f"Cobertura: {combinaciones_reales / combinaciones_teoricas * 100:.2f}%")
print("\nInterpretacion: la mayoria de semanas un SKU NO se vende en un centro especifico.")
print("Esto se tratara expandiendo la serie y rellenando con cero en el feature engineering.")

Combinaciones teoricas (SKU x centro x semana):  1,515,828
Combinaciones con venta:                           184,200
Cobertura: 12.15%

Interpretacion: la mayoria de semanas un SKU NO se vende en un centro especifico.
Esto se tratara expandiendo la serie y rellenando con cero en el feature engineering.


In [14]:
# Estadisticas de cantidad y valor semanal
ventas_semanales[["cantidad_total", "valor_bruto_total", "num_transacciones"]].describe()

,cantidad_total,valor_bruto_total,num_transacciones
count,"184,200.0000","184,200.0000","184,200.0000"
mean,128.5991,"828,398.8388",3.1460
std,"1,095.7964","7,387,916.8165",5.4409
min,0.0170,0.0000,1.0000
25%,1.0000,"14,881.7500",1.0000
50%,4.0000,"56,544.0000",2.0000
75%,20.0000,"221,680.0000",3.0000
max,"63,490.0000","1,030,931,689.0000",190.0000


In [15]:
# Distribucion por centro de operacion
por_centro = (
    ventas_semanales.groupby("centro_operacion")
    .agg(
        filas=("item", "count"),
        skus_distintos=("item", "nunique"),
        cantidad_total=("cantidad_total", "sum"),
        valor_total=("valor_bruto_total", "sum"),
    )
    .sort_values("valor_total", ascending=False)
)
por_centro

,filas,skus_distintos,cantidad_total,valor_total
centro_operacion,,,,
002,79432,2860,"15,634,926.4960","105,103,589,972.0000"
001,87333,2858,"7,487,352.7480","39,374,259,893.0000"
003,17435,2100,"565,678.6100","8,113,216,243.0000"


In [16]:
# Top 20 SKUs por valor bruto total
top_skus = (
    ventas_semanales.groupby("item")["valor_bruto_total"].sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
    .merge(dim_producto[["item", "nombre_item", "nombre_linea_n1"]], on="item", how="left")
)
top_skus

,item,valor_bruto_total,nombre_item,nombre_linea_n1
0,000041,"31,620,995,431.0000",CEMENTO GRIS *50 KL T1 ARGOS,CEMENTOS
1,000291,"10,483,752,049.0000",VARILLA CORR 1/2 * 6 MTS,ACEROS
2,000289,"7,127,240,990.0000",VARILLA CORR 3/8 * 6 MTS,ACEROS
3,009511,"4,040,091,801.0000",SUR CEMENTO ARGOS T1,CEMENTOS
4,000255,"3,766,862,636.0000",HIERRO DE 1/4 *KG P/FLEJES,ACEROS
5,000292,"3,245,896,878.0000",VARILLA CORR 5/8 * 6 MTS,ACEROS
6,010213,"3,155,990,085.0000",T1 CEMENTO GRIS * 50KG RETIRADO EN PLANT,CEMENTOS
7,000551,"2,353,921,226.0000",TEJA PROTEJA P7 6 (1.80),CUBIERTAS
8,000539,"1,892,297,000.0000",PLACA PANEL YESO GYPLAC (1/2 ),CONSTRUCCION LIVIANA
9,000548,"1,843,408,886.0000",TEJA PROTEJA P7 10(3.05),CUBIERTAS


In [17]:
# Concentracion del valor en los top SKUs (anticipo del Pareto)
valor_por_sku = ventas_semanales.groupby("item")["valor_bruto_total"].sum().sort_values(ascending=False)
valor_total = valor_por_sku.sum()

for n in [10, 50, 100, 500, 1000]:
    pct = valor_por_sku.head(n).sum() / valor_total * 100
    print(f"Top {n:>5} SKUs concentran el {pct:>5.2f}% del valor total")

print(f"\nSKUs totales con venta: {len(valor_por_sku):,}")

Top    10 SKUs concentran el 45.57% del valor total
Top    50 SKUs concentran el 68.16% del valor total
Top   100 SKUs concentran el 77.08% del valor total
Top   500 SKUs concentran el 93.13% del valor total
Top  1000 SKUs concentran el 97.11% del valor total

SKUs totales con venta: 4,282


## 5. Resumen y conclusiones iniciales

Hallazgos preliminares del perfil de los datos:

- **Volumen:** 579,485 filas en `ventas_crudas` y 184,200 en `ventas_semanales`, derivadas de 4,282 SKUs distintos en 3 centros de operación.
- **Periodo:** del 1 de enero de 2024 al 30 de marzo de 2026 (119 semanas ISO).
- **Calidad:** las tablas están tipificadas, sin nulos críticos, y la suma de cantidades cuadra al 100% entre `ventas_crudas` y `ventas_semanales`.
- **Esporadicidad:** solo el ~12% de combinaciones (SKU × centro × semana) tiene venta. Confirma que se requerirá expandir la serie con ceros antes del feature engineering.
- **Concentración:** los top SKUs concentran un porcentaje muy alto del valor, lo que anticipa una clasificación ABC pronunciada (se formalizará en el notebook 04).
- **Asimetría entre centros:** los centros 001 y 002 tienen volumen comparable, mientras que el centro 003 es ~10 veces más pequeño.

**Listo para `02_eda.ipynb`**, donde se profundizará con visualizaciones, estacionalidad y análisis temporal.